# Lecture 6 (Dictionaries and sets)

## Exercise 6.1 (histogram)

Write a method histogram that given a list of values, returns a list of pairs (value, frequency).

_Example_. `histogram(['A', 'B', 'A', 'A', 'C', 'E', 'C'])` should return `[('A', 3), ('B', 1), ('C', 2), ('E', 1)]`.

_Hint_. Use a dictionary and the dictionary method `get`.

_Note_. In the standard library module `collections` the `Counter` method implements the same functionality.

    import collections
    histogram = list(collections.Counter(['A', 'B', 'A', 'A', 'C', 'E', 'C']).items())

In [ ]:
def histogram(L):
    ''' given a list of elements, returns a list of pairs (element, frequency) '''

    freq = {}
    for e in L:
        freq[e] = freq.get(e, 0) + 1
    #return list(freq.items())
    return [(e, freq[e]) for e in freq]


print(histogram(['A', 'B', 'A', 'A', 'C', 'E', 'C']))

from collections import Counter
print(Counter(['A', 'B', 'A', 'A', 'C', 'E', 'C']))

## Exercise 6.2 (frequent long words)

_This exercise extends Exercise 6.1._

Write a program that prints the most frequent words containing at least six characters occurring in a text file. E.g. for the [saxo.txt](http://www.gutenberg.org/ebooks/1150.txt.utf-8) file it should print something like the below.

    Rank Freq. Word
    ====================
       1   372 should
       2   221 himself
       3   199 father
       4   196 battle
       5   187 though
       6   181 thought
       7   172 against
       8   142 before
       9   134 daughter
      10   124 country

You can use the following code to split the content of a file into a list of lower case words, ignoring everything that is not a letter in the text.

    import re
    txt = open('saxo.txt', encoding='utf8').read()
    txt = txt.lower()
    words = re.split('[^a-z]+', txt)

_Hint_. Use the `sorted` function to sort a list of tuples of the form (frequency, word), and try to use list comprehension to create such a list from a frequency dictionary.

In [ ]:
# saxo.txt = http://www.gutenberg.org/cache/epub/1150/pg1150.txt

import re

txt = open('saxo.txt', encoding='utf8').read()
txt = txt.lower()
words = re.split('[^a-z]+', txt)

freq = {}
for w in words:
    freq[w] = freq.get(w, 0) + 1

# top = sorted([(freq[w], w) for w in freq if len(w) >= 6])[-1:-11:-1]
top = sorted([(freq[w], w) for w in freq if len(w) >= 6], reverse=True)[:10]

print('Rank Freq. Word')
print('====================')
for idx, (occur, word) in enumerate(top, start=1):
    # print('%4d %5d %s' % (idx, occur, word))
    print(f'{idx:4d} {occur:5d} {word}')

Rank Freq. Word
   1   372 should
   1   372 should
   2   221 himself
   2   221 himself
   3   199 father
   3   199 father
   4   196 battle
   4   196 battle
   5   187 though
   5   187 though
   6   181 thought
   6   181 thought
   7   172 against
   7   172 against
   8   142 before
   8   142 before
   9   134 daughter
   9   134 daughter
  10   124 country
  10   124 country


## Exercise 6.3 - handin 3 (triplet distance - part I)

_This handin together with the next handin constitutes one smaller project. The code from the first project will be used in the second project. In this first project the aim should be to write elegant code using Python's tuples and list comprehensions._

     morphological        18S rDNA
    characteristics     sequence data

       /\                 /\
      /  \               /  \
    'A'  /\            'A'  /\
        /  \               /  \           A = Glycine max
      'C'  /\            'G'  /\          B = Characium perforatum
          /  \               /  \         C = Friedmannia isaelensis
         /    \             /    \        D = Parietochloris pseudoalveolaris
        /\     \           /\     \       E = Dunaliella parva
       /  \    /\         /  \    /\      F = Characium hindakii
     'E'  'G' /  \      'E'  'F' /  \     G = Chlamydomonas
             /    \             /    \
            /\    'D'          /\    'D'
           /  \               /  \
         'B'  'F'           'B'  'C'

    Left:  ('A', ('C', (('E', 'G'), (('B', 'F'), 'D'))))
    Right: ('A', ('G', (('E', 'F'), (('B', 'C'), 'D'))))

_Background_. In this and the next handin we will implement an algorithm to compute the so called _triplet distance_ between two rooted binary trees. The notion of triplet distance was introduced in a paper by Dobson in 1975 and was e.g. considered in the context of Phylogenetic Trees by Critchlow et al. in 1996 (the references are provided to show the scientific background of this exercise - it is not necessary to read the papers to solve this exercise).

* Annette J. Dobson,
  **Comparing the Shapes of Trees**.
  Combinatorial Mathematics III, Lecture Notes in Mathematics, volume 452, 95-100, 1975,
  doi: [10.1007/BFb0069548](https://doi.org/10.1007/BFb0069548).

* Douglas E. Critchlow Dennis K. Pearl Chunlin Qian,
  **The Triples Distance for Rooted Bifurcating Phylogenetic Trees**.
  _Systematic Biology_, 45(3):323-334, 1996,
  doi: [10.1093/sysbio/45.3.323](https://doi.org/10.1093/sysbio/45.3.323).

Above is an example from the paper by Critchlow et al. showing two phylogenies for chloroccalean zoosporic green algae, generated based on morphological characteristics (left) and on 18S rDNA sequence data (right).

The _triplet distance_ (defined below) between the two binary trees is a measure on how different the two resulting trees are. There are many definitions of distance measures between trees - we will in this exercise only consider the triplet distance between two rooted binary trees.

_Tree representations_. We restrict the input to our algorithm to be rooted binary trees, i.e. trees where all internal nodes have exactly two children. We assume that a binary tree is represented by a recursive tuple with leaves being strings, representing the _labels_ of the leaves. For a single tree we require all leaf labels to be distinct, and for two trees to be compared that they have exactly the same set of leaf labels. Below two binary trees are shown with the same leaf labels `'A'`-`'F'`.

    ((('A','F'),'B'),('D',('C','E')))   (((('D','A'),'B'),'F'),('C','E'))

               (a)                                     (b)

                /\                                      /\
               /  \                                    /  \
              /    \                                  /    \
             /      \                                /      \
            /        \                              /        \
           /          \                            /          \
          /\          /\                          /\          /\
         /  \        /  \                        /  \        /  \
        /    \      /    \                      /    \     'C'  'E'
       /\    'B'  'D'    /\                    /      \
      /  \              /  \                  /\      'F'
    'A'  'F'          'C'  'E'               /  \
                                            /    \
                                           /\    'B'
                                          /  \
                                        'D'  'A'

For a tree with _n_ labels the number of subsets containing three labels equals binomial(_n_, 3) = _n_ · (_n_ - 1) · (_n_ - 2) / 6.  Each such set of three labels defines a _triplet_ in each input tree, i.e. the three leaves with the three labels induce a subtree with three leaves. Below we show the triplets induced for the three labels `{'A', 'D', 'F'}`.  The `'*'` marks the lowest common ancestor (LCA) of the three labels. We say that these nodes are the _anchors_ of the triplets in the two trees.

               (a)                                    (b)

         anchor *                                      /\
               / \                                    /  \
              /   \                                  /    \
             /     \                                /      \
            /       \                              /        \
           /         \                            /          \
          /\         /\                   anchor *           /\
         /  \       /  \                        / \         /  \
        /    \     /    \                      /   \      'C'  'E'
       /\    'B' 'D'    /\                    /     \
      /  \             /  \                  /\     'F'
    'A'  'F'         'C'  'E'               /  \
                                           /    \
                                          /\    'B'
                                         /  \
                                       'D'  'A'

                  Induced triplets by {'A', 'D', 'F'}

                /\                                    /\
               /  \                                  /  \
              /    \                                /    \
             /\    'D'                             /\    'F'
            /  \                                  /  \
          'A'  'F'                              'D'  'A'

        (('A', 'F'), 'D')                     (('D', 'A'), 'F')

Since we only care about the topologies of the induced trees, and not if a child is the left or right child of its parent, we for each triplet define its unique _canonical triplet representation_. For a triplet anchored at a node, with label _a_ in one subtree and _b_ and _c_ in the other subtree where _b_ ≤ _c_, we define the canonical representation as the triplet where _a_ is in the left subtree and _b_ and _c_ are in the right subtree, with _b_ to the left of _c_. Below are the canonical triplet representations of the two triplets above:

        Induced canonical triplets by {'A', 'D', 'F'}

           /\                                    /\
          /  \                                  /  \
         /    \                                /    \
       'D'    /\                             'F'    /\
             /  \                                  /  \
           'A'  'F'                              'A'  'D'

    ('D', ('A', 'F'))                     ('F', ('A', 'D'))

**Definition**: Given two trees, where each tree has _n_ distinctly labeled leaves and the two trees have identical label sets, the _triplet distance_ between the two trees equals _n_ · (_n_ - 1) · (_n_ - 2) / 6 minus the number of label subsets of size three with identical induced canonical triplet representations in both trees.

_For each of the following questions try to make efficient use of Python's tuples and list comprehension._

1.  Make a function `generate_labels(n)`, that given an integer `n` returns a list of `n` _distinct strings_, e.g. `'A'`, `'B'`, ... or `'L1'`, `'L2'` ...

    _Example_. `generate_labels(5)` could return `['A', 'B', 'C', 'D', 'E']`.


2.  Make a function `permute(L)`, that given a list `L`, returns a new list containing a _random permutation_ of the elements in `L`.

    _Hint_. Construct the new list left-to-right by randomly selecting an element not selected so far.  To generate a random integer in the interval [a, b], you can you the function `randint(a, b)` from the module [`random`](https://docs.python.org/3/library/random.html) (use `from random import randint` to get access to the function).

    _Note_. Using the functions `shuffle` or `sample` from the module `random` to solve the question would be considered cheating.

    _Example_. `permute(['A', 'B', 'C'])` could return `['B', 'C', 'A']`.


3.  Make a function `pairs(L)`, that given a list of comparable elements, returns a list of all pairs, i.e. tuples with two elements, `(a, b)` where `a` < `b`.

    _Example_. `pairs(['A', 'F', 'B'])` should return `[('A', 'F'), ('A', 'B'), ('B', 'F')]`.


4.  Make a function `canonical_triplets(A, B)` that returns a list of all canonical triples where the left subtree contains a label from `A` and the right subtree is a pair from `B`.

    _Example_. `canonical_triplets(['A', 'B'], ['C', 'D', 'E'])` should return: `[('A', ('C', 'D')), ('A', ('C', 'E')), ('A', ('D', 'E')), ('B', ('C', 'D')), ('B', ('C', 'E')), ('B', ('D', 'E'))]`.


5.  Make a function `anchored_triplets(L, R)` that returns a list of all canonical triples anchored at a node _v_ where the leaves in the left subtree of _v_ contains the labels in the list `L` and the leaves in the right subtree of _v_ contains the labels in the list `R`.

    _Example_. For the root of the tree (a) `anchored_triplets(['A', 'F', 'B'], ['D', 'C', 'E'])` should return the following 18 canonical triplets: `[('A', ('D', 'E')), ('A', ('C', 'D')), ('A', ('C', 'E')), ('F', ('D', 'E')), ('F', ('C', 'D')), ('F', ('C', 'E')), ('B', ('D', 'E')), ('B', ('C', 'D')), ('B', ('C', 'E')), ('D', ('A', 'F')), ('D', ('A', 'B')), ('D', ('B', 'F')), ('C', ('A', 'F')), ('C', ('A', 'B')), ('C', ('B', 'F')), ('E', ('A', 'F')), ('E', ('A', 'B')), ('E', ('B', 'F'))]`.


_Handin format_. As in handin 1 one .py file with a docstring with reflection.

In [ ]:
'''
Compute the TripletDistance between two rooted binary trees, each
represented by nested tuples and distinct strings as leaves.
'''

import math
from random import randint

######################################################################
#                              Handin 3
######################################################################


def generate_labels(n):
    return ([chr(ord('A') + i) for i in range(min(26, n))]
            + ['L' + str(i) for i in range(1, n - 25)])


def permute(L):
    p, n = L[:], len(L)
    for i in range(n):
        j = randint(i, n - 1)
        p[i], p[j] = p[j], p[i]
    return p


def pairs(L):
    return [(x, y) for x in L for y in L if x < y]


def canonical_triplets(A, B):
    return [(a, (b, c)) for b, c in pairs(B) for a in A]


def anchored_triplets(L, R):
    return canonical_triplets(L, R) + canonical_triplets(R, L)


######################################################################
#                              Handin 4
######################################################################


def generate_tree(labels):
    if len(labels) == 1:
        return labels[0]

    i = randint(1, len(labels) - 1)
    left = generate_tree(labels[:i])
    right = generate_tree(labels[i:])

    return (left, right)


def generate_triplets(tree):
    if isinstance(tree, str):
        labels = [tree]
        triplets = []
    else:
        left, right = tree
        left_labels, left_triplets = generate_triplets(left)
        right_labels, right_triplets = generate_triplets(right)

        labels = left_labels + right_labels
        triplets = left_triplets + right_triplets
        triplets += anchored_triplets(left_labels, right_labels)
    # print(f'generate_triplets({tree!r}) = {(labels, triplets)}')
    return (labels, triplets)


def triplet_distance(tree1, tree2):
    labels1, triplets1 = generate_triplets(tree1)
    labels2, triplets2 = generate_triplets(tree2)

    if set(labels1) != set(labels2):
        print('leaf labels differ')
    if not len(labels1) == len(labels2) == len(set(labels2)):
        print('dublicate labels')

    n = len(labels1)
    common = len(set(triplets1) & set(triplets2))

    return n * (n - 1) * (n - 2) // 6 - common


######################################################################
#                              Optional
######################################################################


def print_ascii_tree(tree):
    anchor, lines = ascii_tree(tree)
    print('\n'.join(lines))


def ascii_tree(tree):
    if isinstance(tree, str):
        return 1.0 + len(tree) / 2, ["'" + tree + "'"]

    left, right = tree
    left_anchor, left_lines = ascii_tree(left)
    right_anchor, right_lines = ascii_tree(right)

    left_anchor = math.ceil(left_anchor)
    right_anchor = math.floor(right_anchor)
    left_width = len(left_lines[0])
    right_width = len(right_lines[0])
    spacing = 1 + (left_width - left_anchor + right_anchor + 1) % 2
    line_length = left_width + spacing + right_width
    right_anchor = left_width + spacing + right_anchor

    diff = len(right_lines) - len(left_lines)
    if diff >= 0:
        left_lines += [' ' * left_width for _ in range(diff)]
    else:
        right_lines += [' ' * right_width for _ in range(-diff)]

    lines = [l + ' ' * spacing + r for l, r in zip(left_lines, right_lines)]
    while left_anchor < right_anchor:
        lines.insert(0, ' ' * left_anchor
                        + '/'
                        + ' ' * (right_anchor - left_anchor - 2)
                        + '\\'
                        + ' ' * (line_length - right_anchor))
        left_anchor += 1
        right_anchor -= 1

    return left_anchor, lines


tree1 = ((('A', 'F'), 'B'), ('D', ('C', 'E')))
tree2 = (((('D', 'A'), 'B'), 'F'), ('C', 'E'))
print_ascii_tree(tree1)
print_ascii_tree(tree2)
print('Triplet distance: ', triplet_distance(tree1, tree2))

tree = ((('a', 'b'), 'c'), (('d', 'e'), ('f', 'g')))
print_ascii_tree(tree)
print(generate_triplets(tree))

In [ ]:
'''Performance evaluation of triplet distance function'''

import matplotlib.pyplot as plt
from time import time
from random import randint
import gc

def path(n):
    return 'L1' if n == 1 else (path(n-1), f'L{n}')

def balanced(i, j=None):
    if j == None:
        i, j = 1, i
    m = (i + j) // 2
    return f'L{i}' if i == j else (balanced(i, m), balanced(m + 1, j))

def random_tree(i, j=None):
    if j == None:
        i, j = 1, i
    m = randint(i, j)
    return f'L{i}' if i == j else (balanced(i, m), balanced(m + 1, j))

ns = []
time_path = []
time_balanced = []
time_random = []

ns = range(10, 1000, 10)
for n in ns:
    for times, generator in (
        (time_path, path),
        (time_balanced, balanced),
        (time_random, random_tree)
    ):
        if times and times[-1] > 10:
            continue
        print(n, generator.__name__)
        tree = generator(n)
        gc.collect
        start = time()
        dist = triplet_distance(tree, tree)
        end = time()
        times.append(end - start)

for times, label in (
    (time_path, 'path'),
    (time_random, 'random tree'),
    (time_balanced, 'balanced tree')
):
    plt.plot(ns[:len(times)], times, '.-', label=label)
plt.legend()
plt.xlabel('Tree size')
plt.ylabel('Time (seconds)')
plt.title('Time for computing triplet distance (with itself)')
plt.show()

In [ ]:
'''
Compute the TripletDistance between two rooted binary trees, each
represented by nested tuples and distinct strings as leaves.
'''


def compute(tree):
    '''For a given tree return tuple(leaves, triplets) with two lists.'''

    def pairs(l):
        return [(x, y) for x in l for y in l if x < y]

    if not isinstance(tree, tuple):
        return [tree], []

    left, right = tree

    left_leaves, left_triplets = compute(left)
    right_leaves, right_triplets = compute(right)

    leaves = left_leaves + right_leaves
    triplets = left_triplets + right_triplets
    triplets += [(l, p) for l in left_leaves for p in pairs(right_leaves)]
    triplets += [(l, p) for l in right_leaves for p in pairs(left_leaves)]

    return leaves, triplets


def triplet_distance(tree1, tree2):
    leaves1, triplets1 = compute(tree1)
    leaves2, triplets2 = compute(tree2)

    if set(leaves1) != set(leaves2) or not len(set(leaves1)) == len(leaves1) == len(leaves2):
        print('Not common leaf labels')
    else:
        n = len(leaves1)
        common_triplets = set(triplets1) & set(triplets2)
        return n * (n - 1) * (n - 2) // 3 // 2 - len(common_triplets)


tree1 = ((('A', 'F'), 'B'), ('D', ('C', 'E')))
tree2 = (((('D', 'A'), 'B'), 'F'), ('C', 'E'))
print('Triplet distance: ', triplet_distance(tree1, tree2))

In [ ]:
'''A compact solution computing the triplet distance between two trees O(n^4)'''

def triplet_distance(tree1, tree2):
    def compute(tree):
        if not isinstance(tree, tuple): return [tree], []
        (ll, lt), (rl, rt) = [compute(c) for c in tree]
        t = [(x, (y, z)) for a, b in [(ll, rl), (rl, ll)] for y in b for z in b if y < z for x in a]
        return ll + rl, lt + rt + t

    (leaves1, triplets1), (leaves2, triplets2) = compute(tree1), compute(tree2)
    return len(set(triplets1) - set(triplets2))

tree1 = ((('A', 'F'), 'B'), ('D', ('C', 'E')))
tree2 = (((('D', 'A'), 'B'), 'F'), ('C', 'E'))
print('Triplet distance: ', triplet_distance(tree1, tree2))

In [ ]:
'''Computing the triplet distance by collecting triplets in a single list O(n^3).'''

def triplet_distance(tree1, tree2):
    def compute(tree, triplets):
        if not isinstance(tree, tuple): return [tree]
        ll, rl = [compute(c, triplets) for c in tree]
        triplets += [(x, (y, z)) for a, b in ((ll, rl), (rl, ll)) for y in b for z in b if y < z for x in a]
        return ll + rl

    triplets1, triplets2 = [], []
    compute(tree1, triplets1)
    compute(tree2, triplets2)
    return len(set(triplets1) - set(triplets2))

tree1 = ((('A', 'F'), 'B'), ('D', ('C', 'E')))
tree2 = (((('D', 'A'), 'B'), 'F'), ('C', 'E'))
print('Triplet distance: ', triplet_distance(tree1, tree2))

In [ ]:
'''
Different ways to implement permute, including slow and non-uniform.
'''

import matplotlib.pyplot as plt
from time import time
from random import randint


def permute_swap(L):
    '''Random swap with random element from prefix. Time n.'''

    L = L.copy()
    for i in range(len(L))[::-1]:
        j = randint(0, i)
        L[i], L[j] = L[j], L[i]
    return L


def permute_pop(L):
    '''Permute using expensive list.pop. Expected time n**2.'''

    L = L.copy()
    return [L.pop(randint(0, i)) for i in range(len(L))[::-1]]


def permute_try(L):
    '''Permute using repeated sampling without removal. Expected n**2*log n.'''

    shuffled = []
    while len(shuffled) < len(L):
        e = L[randint(0, len(L) - 1)]
        if e not in shuffled:  # success probability (|L| - |shuffled|) / |L|
            shuffled.append(e)
    return shuffled


def permute_random_swap(L):
    '''Shuffle that does NOT generate uniform permutation.'''

    n = len(L)
    L = L.copy()
    for i in range(n):
        j = randint(0, n - 1)  # swap with random position in L
        L[i], L[j] = L[j], L[i]
    return L


def test_distribution(permute_function, n=3, repeats=1000):
    '''Test if shuffle function outputs are at a random position.'''

    fq = [[0] * n for _ in range(n)]
    for _ in range(repeats):
        L = list(range(n))
        permutation = permute_function(L)
        for position, value in enumerate(permutation):
            fq[position][value] += 1

    print(f'output position \ input position = frequency, {permute_function.__name__}, {n=}')
    for position in range(n):
        for value in range(n):
            print(f'{fq[position][value] / repeats:.3f}', end=' ')
        print()
    print()


def distribution_random_swap(n=3):
    '''Compute theoretical distribution for permute_random_swap.'''

    pr = [[1 if pos == value else 0 for value in range(n)] for pos in range(n)]
    for k in range(n):
      pr = [[pr[pos][value] * (n - 1) / n + pr[k][value] * 1 / n if pos != k else 1 / n
             for value in range(n)] for pos in range(n)]
    print(f'theoretical distribution, permute_random_swap, {n=}')
    print('output position \ input position = probability')
    for position in range(n):
        for value in range(n):
            print(f'{pr[position][value]:.3f}', end=' ')
        print()
    print()


def permute_performance(permute_functions):
    '''Plot performance of various shuffle functions.'''

    for shuffle in permute_functions:
        N = []
        T = []
        n = 100
        print(shuffle.__name__, end=' ')
        while True:
            print(n, end= ' ')
            total_time = 0
            for k in range(1, 100):
                L = list(range(n))
                start = time()
                shuffle(L)
                end = time()
                t = end - start
                total_time += t
                if total_time >= 0.5:
                    break
            N.append(n)
            T.append(total_time / k)
            if t > 1:  # max time
                break
            n = n * 3 // 2
        print()
        plt.plot(N, T, '.-', label=shuffle.__name__)
    plt.legend()
    plt.xlabel('list length')
    plt.ylabel('time (seconds)')
    plt.show()


test_distribution(permute_swap, n=3, repeats=100_000)
test_distribution(permute_random_swap, n=3, repeats=100_000)
distribution_random_swap(n=3)
permute_performance([permute_try, permute_pop, permute_swap])